In [20]:
from glob import glob
import os
from tqdm import tqdm
from PIL import Image
from utils import rajouter_features

In [21]:
crop_size = 32
split = "val"   # choisir train, val ou test
img_type = "annotations"   # choisir sat ou annotations

input_dir = f"data/dataset/{split}"
output_dir = f"data/dataset_crops_{img_type}/{split}_{crop_size}x{crop_size}"

if img_type == "sat":
    path = "S2L2A"
elif img_type == "annotations":
    path = "Esri10_LC"
else:
    print("erreur type d'image mal définit")

# list des chemins vers les images
image_paths = [
    p for p in glob(os.path.join(input_dir, "**", f"*{path}*"), recursive=True)
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(len(image_paths), "images found.")

os.makedirs(output_dir, exist_ok=True)

53 images found.


In [22]:
def get_label(pixel):
    if pixel == (0, 0, 0):
        return "0"
    elif pixel == (181, 254, 142):
        return "1"
    elif pixel == (254, 217, 142):
        return "2"
    elif pixel == (254, 153, 41):
        return "3"
    elif pixel == (204, 76, 2):
        return "4"
    elif pixel == (255, 255, 255):
        return "0"
    else:
        return print("erreur, unknown label:", pixel)

In [23]:
for path_to_img in tqdm(image_paths):

    # charger l'image
    img = Image.open(path_to_img).convert("RGB")

    # charger le masque de brûlé
    path_to_img_gra = path_to_img.replace(path, "GRA")
    path_to_img_del = path_to_img.replace(path, "DEL")
    if os.path.exists(path_to_img_gra):
        path_to_img_burned = path_to_img_gra
    elif os.path.exists(path_to_img_del):
        path_to_img_burned = path_to_img_del
    else:
        print("********** image brûlée non trouvée *********")
        continue
    img_burn = Image.open(path_to_img_burned).convert("RGB")

    # rajouter des features artificielles
    if img_type == "annotations":
        img = rajouter_features(img, img_burn)


    width, height = img.size

    # Calculer combien de crops on peut faire horizontalement et verticalement
    n_crops_x = width // crop_size
    n_crops_y = height // crop_size

    base_name = os.path.splitext(os.path.basename(path_to_img))[0]

    # Découper en crops 32x32
    for i in range(n_crops_y):
        for j in range(n_crops_x):
            left = j * crop_size
            upper = i * crop_size
            right = left + crop_size
            lower = upper + crop_size

            crop = img.crop((left, upper, right, lower))

            crop_burn = img_burn.crop((left, upper, right, lower))
            middle_pixel = crop_burn.getpixel((crop_size // 2, crop_size // 2))
            label = get_label(middle_pixel)

            # Création du dossier de la classe si besoin
            label_dir = os.path.join(output_dir, label)
            os.makedirs(label_dir, exist_ok=True)

            # Nom unique pour le crop
            crop_filename = f"{base_name}_crop_{i}_{j}.png"
            crop.save(os.path.join(label_dir, crop_filename))


100%|██████████| 53/53 [01:37<00:00,  1.84s/it]
